# **TAKE HOME TEST PROJECT - NOTEBOOK 3**

# Dashboard Development

```
Name: Akmal Zuhdy Prasetya
Class: Data Analytics & Business Intelligence
Batch: 19
```

## Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## Tahap 5 — Pembuatan Dashboard

### 5.1 Dashboard Planning & Storyline

#### **Dashboard Narrative**

Bisnis TheLook mengalami pertumbuhan yang sangat cepat, tetapi masih menghadapi risiko revenue leakage yang signifikan akibat tingginya return behavior pada beberapa kategori fashion utama.

#### **Dashboard Structure**

**PAGE 1 — Executive Overview**

Purpose: Memberikan gambaran umum performa bisnis.

KPI yang akan digunakan:

| KPI                      | Description            |
| ------------------------ | ---------------------- |
| Total Revenue            | total monetization     |
| Total Orders             | total order volume     |
| Total Customers          | total unique customers |
| Return Rate              | % returned orders      |
| Returned Revenue         | leakage value          |
| Avg Fulfillment Duration | operational efficiency |

Catatan visual:

| Visual                      | Type           |
| --------------------------- | -------------- |
| Monthly Revenue Trend       | Line Chart     |
| Monthly Customer Activity   | Line Chart     |
| Revenue vs Returned Revenue | Stacked Bar    |
| Top Revenue Categories      | Horizontal Bar |

**PAGE 2 — Customer & Acquisition Analysis**

Purpose: Menganalisis traffic acquisition dan customer engagement.

Catatan visual:

| Visual                            | Type        |
| --------------------------------- | ----------- |
| Traffic Source Distribution       | Donut Chart |
| Conversion Rate by Traffic Source | Bar Chart   |
| Customer Activity by Hour         | Line Chart  |
| Monthly Activity Trend            | Line Chart  |

Catatan filter:

| Filter         |
| -------------- |
| Year           |
| Traffic Source |

**PAGE 3 — Revenue & Product Performance**

Purpose: Menganalisis monetization dan category performance.

Catatan visual:

| Visual                        | Type         |
| ----------------------------- | ------------ |
| Top Revenue Categories        | Bar Chart    |
| Top Profit Categories         | Bar Chart    |
| Profit Margin by Category     | Bar Chart    |
| Revenue Trend                 | Line Chart   |
| Revenue vs Profit by Category | Scatter Plot |

Catatan filter:

| Filter     |
| ---------- |
| Category   |
| Department |
| Year       |

**PAGE 4 — Returns & Operational Insights**

Purpose: Menganalisis leakage risk dan operational efficiency.

Catatan visual:

| Visual                               | Type       |
| ------------------------------------ | ---------- |
| Return Rate Trend                    | Line Chart |
| Returned Revenue by Category         | Bar Chart  |
| Return Rate by Category              | Bar Chart  |
| Fulfillment Duration Trend           | Line Chart |
| Returned vs Non-Returned Fulfillment | Bar Chart  |

Catatan filter:

| Filter        |
| ------------- |
| Category      |
| Year          |
| Return Status |


### 5.2 Prepare Dashboard Datasets

**DATASET 1 — executive_kpi.csv**

Purpose: Digunakan untuk KPI cards di Executive Overview.

| Metric                   |
| ------------------------ |
| Total Revenue            |
| Total Orders             |
| Total Customers          |
| Return Rate              |
| Returned Revenue         |
| Avg Fulfillment Duration |

**DATASET 2 — monthly_revenue.csv**

Purpose: Revenue trend analysis.

| Column        |
| ------------- |
| order_year    |
| order_month   |
| total_revenue |

**DATASET 3 — monthly_activity.csv**

Purpose: Customer activity trend.

| Column       |
| ------------ |
| event_year   |
| event_month  |
| total_events |

**DATASET 4 — traffic_source_analysis.csv**

Purpose: Customer acquisition analysis.

| Column              |
| ------------------- |
| traffic_source      |
| total_events        |
| total_sessions      |
| purchase_events     |
| conversion_rate_pct |

**DATASET 5 — hourly_activity.csv**

Purpose: Customer temporal behavior.

| Column       |
| ------------ |
| event_hour   |
| total_events |

**DATASET 6 — category_revenue_profit.csv**

Purpose: Revenue & profitability analysis.

| Column            |
| ----------------- |
| category          |
| total_revenue     |
| total_profit      |
| profit_margin_pct |

**DATASET 7 — category_returns.csv**

Purpose: Return risk analysis.

| Column          |
| --------------- |
| category        |
| total_items     |
| returned_items  |
| return_rate_pct |

**DATASET 8 — category_leakage.csv**

Purpose: Returned revenue analysis.

| Column           |
| ---------------- |
| category         |
| returned_revenue |

**DATASET 9 — monthly_returns.csv**

Purpose: Return trend analysis.

| Column          |
| --------------- |
| order_year      |
| order_month     |
| return_rate_pct |

**DATASET 10 — monthly_fulfillment.csv**

Purpose: Operational performance trend.

| Column                   |
| ------------------------ |
| order_year               |
| order_month              |
| avg_fulfillment_duration |

#### a. Import Cleaned Datasets

In [ ]:
path = '/content/drive/MyDrive/Bootcamp DABI Batch 19+/program/take home test/cleaned_data/'

df_users = pd.read_csv(path + 'users_cleaned.csv')
df_products = pd.read_csv(path + 'products_cleaned.csv')
df_orders = pd.read_csv(path + 'orders_cleaned.csv')
df_order_items = pd.read_csv(path + 'order_items_cleaned.csv')
df_events = pd.read_csv(path + 'events_cleaned.csv')

**Datetime Conversion**

In [ ]:
date_columns_orders = [
  'created_at',
  'shipped_at',
  'delivered_at',
  'returned_at'
]

for col in date_columns_orders:
  df_orders[col] = pd.to_datetime(
    df_orders[col],
    format='ISO8601',
    errors='coerce'
  )

date_columns_order_items = [
  'created_at',
  'shipped_at',
  'delivered_at',
  'returned_at'
]

for col in date_columns_order_items:
  df_order_items[col] = pd.to_datetime(
    df_order_items[col],
    format='ISO8601',
    errors='coerce'
  )

df_events['created_at'] = pd.to_datetime(
  df_events['created_at'],
  format='ISO8601',
  errors='coerce'
)

#### b. Recreate Required Features

In [ ]:
# Order Year & Month
df_order_items['order_year'] = (
  df_order_items['created_at'].dt.year
)

df_order_items['order_month'] = (
  df_order_items['created_at'].dt.month
)

# Event Year & Month
df_events['event_year'] = (
  df_events['created_at'].dt.year
)

df_events['event_month'] = (
  df_events['created_at'].dt.month
)

# Event Hour
df_events['event_hour'] = (
  df_events['created_at'].dt.hour
)

# Return Flag
df_order_items['is_returned'] = (
  df_order_items['returned_at']
  .notna()
  .astype(int)
)

# Fulfillment Duration
df_order_items['fulfillment_duration_days'] = (
  (
    df_order_items['delivered_at'] -
    df_order_items['created_at']
  ).dt.days
)

# Remove invalid fulfillment duration
df_order_items = df_order_items[
  df_order_items['fulfillment_duration_days'] >= 0
]

#### **c. Create & Export Dashboard Datasets**

In [ ]:
path_dashboard_data = '/content/drive/MyDrive/Bootcamp DABI Batch 19+/program/take home test/dashboard/dashboard_data/'

##### **DATASET 1 — executive_kpi.csv**

In [ ]:
total_revenue = (
  df_order_items['sale_price']
  .sum()
)

total_orders = (
  df_orders['order_id']
  .nunique()
)

total_customers = (
  df_users['id']
  .nunique()
)

return_rate = (
  df_order_items['is_returned']
  .mean()
) * 100

returned_revenue = (
  df_order_items[
    df_order_items['is_returned'] == 1]['sale_price']
    .sum()
)

avg_fulfillment_duration = (
  df_order_items['fulfillment_duration_days']
  .mean()
)

executive_kpi = pd.DataFrame({
  'metric': [
    'Total Revenue',
    'Total Orders',
    'Total Customers',
    'Return Rate',
    'Returned Revenue',
    'Avg Fulfillment Duration'
  ],
  'value': [
      total_revenue,
      total_orders,
      total_customers,
      return_rate,
      returned_revenue,
      avg_fulfillment_duration
  ]
})

executive_kpi

,metric,value
0,Total Revenue,3.464630e+06
1,Total Orders,1.252260e+05
2,Total Customers,1.000000e+05
3,Return Rate,2.862734e+01
4,Returned Revenue,1.003088e+06
5,Avg Fulfillment Duration,2.990388e+00


**Export Dataset 1**

In [ ]:
executive_kpi.to_csv(
  path_dashboard_data + 'executive_kpi.csv',
  index=False
)

##### **DATASET 2 — monthly_revenue.csv**

In [ ]:
monthly_revenue = (
  df_order_items
  .groupby(
    ['order_year', 'order_month']
  )['sale_price']
  .sum()
  .reset_index()
)

monthly_revenue.rename(
  columns={
    'sale_price': 'total_revenue'
  },
  inplace=True
)

monthly_revenue

,order_year,order_month,total_revenue
0,2019,1,684.000008
1,2019,2,905.210003
2,2019,3,1652.500002
3,2019,4,2495.740010
4,2019,5,4985.079998
...,...,...,...
56,2023,9,151280.470274
57,2023,10,172580.040215
58,2023,11,194038.500224
59,2023,12,227021.240216


**Export Dataset 2**

In [ ]:
monthly_revenue.to_csv(
  path_dashboard_data + '/monthly_revenue.csv',
  index=False
)

##### **DATASET 3 — monthly_activity.csv**

In [ ]:
monthly_activity = (
  df_events
  .groupby(
    ['event_year', 'event_month']
  )['id']
  .count()
  .reset_index()
)

monthly_activity.rename(
  columns={
    'id': 'total_events'
  },
  inplace=True
)

monthly_activity

,event_year,event_month,total_events
0,2019,1,18059
1,2019,2,17594
2,2019,3,19801
3,2019,4,19846
4,2019,5,20663
...,...,...,...
56,2023,9,75061
57,2023,10,83393
58,2023,11,89113
59,2023,12,106248


**Export Dataset 3**

In [ ]:
monthly_activity.to_csv(
  path_dashboard_data + '/monthly_activity.csv',
  index=False
)

##### **DATASET 4 — traffic_source_analysis.csv**

In [ ]:
traffic_source_analysis = (
  df_events
  .groupby('traffic_source')
  .agg(
    total_events=('id', 'count'),
    total_sessions=('session_id', 'nunique'),
    purchase_events=(
      'event_type',
      lambda x: (x == 'purchase').sum()
    )
  )
  .reset_index()
)

traffic_source_analysis[
  'conversion_rate_pct'
] = (
    traffic_source_analysis['purchase_events'] /
    traffic_source_analysis['total_events']
) * 100

traffic_source_analysis

,traffic_source,total_events,total_sessions,purchase_events,conversion_rate_pct
0,Adwords,731144,205010,54542,7.459816
1,Email,1091988,306313,81706,7.482317
2,Facebook,243834,67933,18305,7.507157
3,Organic,122059,34301,9124,7.475074
4,YouTube,242938,68202,18082,7.443051


**Export Dataset 4**

In [ ]:
traffic_source_analysis.to_csv(
  path_dashboard_data + '/traffic_source_analysis.csv',
  index=False
)

##### **DATASET 5 — hourly_activity.csv**

In [ ]:
hourly_activity = (
  df_events
  .groupby('event_hour')['id']
  .count()
  .reset_index()
)

hourly_activity.rename(
  columns={
    'id': 'total_events'
  },
  inplace=True
)

hourly_activity

,event_hour,total_events
0,0,102664
1,1,127670
2,2,124750
3,3,126869
4,4,126076
5,5,126812
6,6,128493
7,7,130242
8,8,128164
9,9,129649


**Export Dataset 5**

In [ ]:
hourly_activity.to_csv(
  path_dashboard_data + '/hourly_activity.csv',
  index=False
)

##### **DATASET 6 — category_revenue_profit.csv**

In [ ]:
category_revenue_profit = (
  df_order_items
    .merge(
      df_products,
      left_on='product_id',
      right_on='id',
      how='left'
    )
    .groupby('category')
    .agg(
      total_revenue=('sale_price', 'sum'),
      total_cost=('cost', 'sum')
    )
    .reset_index()
)

category_revenue_profit['total_profit'] = (
  category_revenue_profit['total_revenue'] -
  category_revenue_profit['total_cost']
)

category_revenue_profit[
  'profit_margin_pct'
] = (
    category_revenue_profit['total_profit'] /
    category_revenue_profit['total_revenue']
) * 100

category_revenue_profit = (
  category_revenue_profit
  .sort_values(
    by='total_revenue',
    ascending=False
  )
)

category_revenue_profit

,category,total_revenue,total_cost,total_profit,profit_margin_pct
11,Outerwear & Coats,418753.569782,186200.631901,232552.937881,55.534557
7,Jeans,399817.320625,214018.099174,185799.221451,46.471029
22,Sweaters,270800.810043,130341.998268,140458.811776,51.867944
21,Suits & Sport Coats,213814.839830,85875.723562,127939.116268,59.836406
23,Swim,207317.160216,105767.560593,101549.599623,48.982727
5,Fashion Hoodies & Sweatshirts,205257.400237,106438.654244,98818.745993,48.143816
17,Sleep & Lounge,169975.880470,82361.047338,87614.833132,51.545450
15,Shorts,163765.140503,81980.425799,81784.714705,49.940246
1,Active,156511.730120,65668.530807,90843.199313,58.042422
24,Tops & Tees,154367.620554,86299.507081,68068.113473,44.094813


**Export Dataset 6**

In [ ]:
category_revenue_profit.to_csv(
  path_dashboard_data + '/category_revenue_profit.csv',
  index=False
)

##### **DATASET 7 — category_returns.csv**

In [ ]:
category_returns = (
  df_order_items
  .merge(
    df_products,
    left_on='product_id',
    right_on='id',
    how='left'
  )
  .groupby('category')
  .agg(
    total_items=('id_x', 'count'),
    returned_items=('is_returned', 'sum')
  )
  .reset_index()
)

category_returns['return_rate_pct'] = (
  category_returns['returned_items'] /
  category_returns['total_items']
) * 100

category_returns = (
  category_returns
  .sort_values(
    by='return_rate_pct',
    ascending=False
  )
)

category_returns

,category,total_items,returned_items,return_rate_pct
3,Clothing Sets,57,18,31.578947
21,Suits & Sport Coats,1628,507,31.142506
20,Suits,329,101,30.699088
10,Maternity,1599,480,30.018762
8,Jumpsuits & Rompers,332,99,29.819277
15,Shorts,3518,1036,29.448550
1,Active,2978,875,29.382136
4,Dresses,1702,498,29.259694
6,Intimates,4232,1236,29.206049
12,Pants,2419,706,29.185614


**Export Dataset 7**

In [ ]:
category_returns.to_csv(
  path_dashboard_data + '/category_returns.csv',
  index=False
)

##### **DATASET 8 — category_leakage.csv**

In [ ]:
category_leakage = (
  df_order_items[
    df_order_items['is_returned'] == 1
  ]
  .merge(
    df_products,
    left_on='product_id',
    right_on='id',
    how='left'
  )
  .groupby('category')['sale_price']
  .sum()
  .reset_index()
)

category_leakage.rename(
  columns={
    'sale_price': 'returned_revenue'
  },
  inplace=True
)

category_leakage = (
  category_leakage
  .sort_values(
    by='returned_revenue',
    ascending=False
  )
)

category_leakage

,category,returned_revenue
11,Outerwear & Coats,122664.410027
7,Jeans,111904.790108
22,Sweaters,75385.829964
21,Suits & Sport Coats,66244.219972
23,Swim,59365.540076
5,Fashion Hoodies & Sweatshirts,59188.330063
15,Shorts,50967.720155
17,Sleep & Lounge,45858.770119
1,Active,45603.890085
4,Dresses,43640.650069


**Export Dataset 8**

In [ ]:
category_leakage.to_csv(
  path_dashboard_data + '/category_leakage.csv',
  index=False
)

##### **DATASET 9 — monthly_returns.csv**

In [ ]:
monthly_returns = (
  df_order_items
  .groupby(
    ['order_year', 'order_month']
  )['is_returned']
  .mean()
  .reset_index()
)

monthly_returns['return_rate_pct'] = (
  monthly_returns['is_returned'] * 100
)

monthly_returns

,order_year,order_month,is_returned,return_rate_pct
0,2019,1,0.300000,30.000000
1,2019,2,0.333333,33.333333
2,2019,3,0.366667,36.666667
3,2019,4,0.265306,26.530612
4,2019,5,0.274725,27.472527
...,...,...,...,...
56,2023,9,0.307753,30.775285
57,2023,10,0.296151,29.615110
58,2023,11,0.286244,28.624420
59,2023,12,0.284926,28.492647


**Export Dataset 9**

In [ ]:
monthly_returns.to_csv(
  path_dashboard_data + '/monthly_returns.csv',
  index=False
)

##### **DATASET 10 — monthly_fulfillment.csv**

In [ ]:
monthly_fulfillment = (
  df_order_items
  .groupby(
    ['order_year', 'order_month']
  )['fulfillment_duration_days']
  .mean()
  .reset_index()
)

monthly_fulfillment.rename(
  columns={
    'fulfillment_duration_days':
    'avg_fulfillment_duration'
  },
  inplace=True
)

monthly_fulfillment

,order_year,order_month,avg_fulfillment_duration
0,2019,1,3.000000
1,2019,2,3.266667
2,2019,3,3.133333
3,2019,4,3.285714
4,2019,5,2.890110
...,...,...,...
56,2023,9,3.045651
57,2023,10,3.000356
58,2023,11,2.988872
59,2023,12,2.988971


**Export Dataset 10**

In [ ]:
monthly_fulfillment.to_csv(
  path_dashboard_data + '/monthly_fulfillment.csv',
  index=False
)

In [ ]:
import os

os.listdir(path_dashboard_data)

['executive_kpi.csv',
 'monthly_revenue.csv',
 'monthly_activity.csv',
 'traffic_source_analysis.csv',
 'hourly_activity.csv',
 'category_revenue_profit.csv',
 'category_returns.csv',
 'category_leakage.csv',
 'monthly_returns.csv',
 'monthly_fulfillment.csv']

### 5.3 Dashboard Creation (Check PBI File)

## Tahap 6 — Rekomendasi Insight Bisnis

### 6.1 Customer Acquisition Strategy

#### **Key Finding**

Hasil analisis menunjukkan bahwa aktivitas customer TheLook sangat didominasi oleh traffic source Email yang berkontribusi hampir 45% dari total customer activity. Namun, conversion rate antar seluruh traffic source relatif homogen di kisaran 7.4%–7.5%, menunjukkan bahwa tidak terdapat channel acquisition yang secara signifikan outperform dibanding channel lainnya.

Selain itu, monthly customer activity menunjukkan tren pertumbuhan yang sangat kuat dari tahun ke tahun, terutama sepanjang 2023 hingga awal 2024, menandakan peningkatan engagement platform secara konsisten.

#### **Business Implication**

Ketergantungan traffic yang terlalu besar pada Email menunjukkan adanya: **acquisition concentration risk**.

Jika performa Email campaign mengalami penurunan, maka aktivitas customer secara keseluruhan berpotensi terdampak signifikan.

Di sisi lain, karena conversion efficiency antar channel relatif serupa, hal ini mengindikasikan bahwa channel alternatif seperti Facebook, YouTube, dan Organic sebenarnya memiliki potensi scaling yang masih belum dimanfaatkan secara optimal.

Pertumbuhan customer activity yang terus meningkat juga menunjukkan bahwa TheLook memiliki momentum customer engagement yang kuat dan dapat dimanfaatkan untuk memperluas customer acquisition strategy secara lebih agresif.

#### **Recommendation**

TheLook disarankan untuk melakukan: **channel diversification strategy**.

Dilakukan dengan mengalokasikan sebagian investasi acquisition dari Email ke channel lain yang memiliki conversion efficiency serupa seperti Facebook, YouTube, dan Organic traffic.

Selain itu, perusahaan juga dapat:

- meningkatkan retargeting campaign pada paid channels,
- mengoptimalkan personalized marketing content,
- serta memperkuat omnichannel acquisition strategy untuk mengurangi ketergantungan pada satu sumber traffic utama.

Untuk mempertahankan growth momentum, TheLook juga dapat memanfaatkan behavioral engagement trend untuk:

- meningkatkan campaign timing,
- audience segmentation,
- dan personalized customer communication berdasarkan activity pattern customer.

#### **Expected Business Impact**

Implementasi strategi diversifikasi acquisition berpotensi:

- mengurangi acquisition concentration risk,
- meningkatkan channel scalability,
- memperluas customer reach,
- serta menciptakan customer acquisition structure yang lebih stabil dan sustainable.

Optimalisasi retargeting dan personalization juga berpotensi meningkatkan customer engagement efficiency tanpa harus bergantung pada peningkatan acquisition spending secara signifikan.

### 6.2 Product & Revenue Optimization

#### **Key Finding**

Hasil analisis menunjukkan bahwa revenue TheLook sangat terkonsentrasi pada beberapa kategori utama, khususnya:

- Outerwear & Coats,
- Jeans,
- Sweaters,
- dan Suits & Sport Coats.

Namun, analisis profitabilitas menunjukkan bahwa kategori dengan revenue tertinggi tidak selalu memiliki profit margin tertinggi. Beberapa kategori seperti:

- Blazers & Jackets,
- Skirts,
- dan Accessories

menunjukkan profit margin yang lebih tinggi dibanding beberapa kategori high-revenue lainnya.

Selain itu, scatter plot revenue vs profit menunjukkan adanya perbedaan efisiensi profit antar kategori produk, yang mengindikasikan bahwa pertumbuhan revenue belum tentu diikuti oleh optimalisasi profitability secara proporsional.

#### **Business Implication**

Kondisi ini menunjukkan bahwa strategi bisnis TheLook saat ini kemungkinan masih **terlalu revenue-oriented**. Sementara itu, **profit efficiency antar kategori belum sepenuhnya dioptimalkan**.

Ketergantungan revenue pada beberapa kategori utama juga menciptakan **category concentration risk**, di mana penurunan performa pada kategori dominan dapat memberikan dampak signifikan terhadap total business performance.

Di sisi lain, keberadaan kategori dengan margin tinggi menunjukkan adanya peluang untuk meningkatkan profitability tanpa harus selalu bergantung pada peningkatan sales volume secara besar-besaran.

#### **Recommendation**

TheLook disarankan untuk mulai menerapkan: **profitability-driven category strategy**.

Dilakukan dengan memprioritaskan pengembangan kategori yang memiliki:

- profit margin tinggi,
- profit efficiency baik,
- dan potensi scalability yang masih besar.

Perusahaan juga dapat:

- melakukan pricing optimization,
- meningkatkan exposure kategori high-margin pada campaign marketing,
- serta mengevaluasi product mix untuk mengurangi ketergantungan terhadap kategori revenue-dominant tertentu.

Untuk kategori high-revenue namun margin relatif lebih rendah, TheLook dapat melakukan:

- cost efficiency evaluation,
- supplier optimization,
- atau bundling strategy

untuk meningkatkan overall profitability.

Selain itu, category performance monitoring secara berkala perlu dilakukan untuk memastikan bahwa growth strategy tidak hanya berfokus pada revenue growth, tetapi juga **sustainable profit growth**.

#### **Expected Business Impact**

Implementasi profitability-driven strategy berpotensi:

- meningkatkan overall business profitability,
- mengurangi category concentration risk,
- menciptakan revenue structure yang lebih sehat,
- serta meningkatkan business sustainability dalam jangka panjang.

Optimalisasi category mix dan margin efficiency juga dapat membantu perusahaan memperoleh **profit growth yang lebih stabil tanpa harus selalu meningkatkan acquisition cost atau sales volume secara agresif**.

### 6.3 Return Leakage Reduction

#### **Key Finding**

Hasil analisis menunjukkan bahwa TheLook memiliki **overall return rate sebesar ~28.6%**, dengan estimasi revenue leakage hampir 29% dari total revenue.

Revenue leakage terbesar berasal dari kategori-kategori utama seperti:

- Outerwear & Coats,
- Jeans,
- Sweaters,
- dan Suits & Sport Coats,

yang juga merupakan kontributor revenue terbesar perusahaan.

Selain itu, beberapa kategori menunjukkan return rate yang relatif tinggi secara konsisten, mengindikasikan bahwa return behavior bukan merupakan isolated issue, melainkan **recurring business challenge**.

Analisis tren bulanan juga menunjukkan bahwa return rate cenderung stabil sepanjang waktu, menandakan bahwa permasalahan return belum mengalami perbaikan signifikan secara struktural.

#### **Business Implication**

Tingkat return dan revenue leakage yang tinggi menunjukkan adanya **significant profitability risk** bagi perusahaan.

Karena sebagian besar leakage berasal dari kategori high-revenue, maka peningkatan sales volume saja tidak akan cukup untuk mengoptimalkan business performance apabila return behavior tetap tinggi.

Kondisi ini juga mengindikasikan adanya kemungkinan:

- product expectation mismatch,
- sizing inconsistency,
- product quality issue,
- atau kurang optimalnya product information yang diterima customer sebelum purchase.

Jika tidak ditangani, tingginya return behavior berpotensi:

- menurunkan effective profitability,
- meningkatkan operational cost,
- memperbesar reverse logistics burden,
- serta mengurangi customer lifetime value dalam jangka panjang.

#### **Recommendation**

TheLook disarankan untuk menerapkan: **return prevention strategy**.

Dilakukan dengan fokus utama pada kategori yang memiliki:

- return contribution tinggi,
- revenue leakage besar,
- dan return rate yang konsisten tinggi.

Beberapa strategi yang dapat dilakukan antara lain:

- meningkatkan kualitas product description,
- memperjelas sizing guide,
- menambahkan customer review dan visual product reference,
- serta melakukan quality control lebih ketat pada kategori high-return.

Perusahaan juga dapat melakukan:

- return pattern analysis,
- customer feedback monitoring,
- dan segmentation terhadap return behavior untuk mengidentifikasi faktor dominan penyebab return pada masing-masing kategori.

Untuk kategori dengan leakage sangat besar, TheLook dapat mempertimbangkan:

- targeted quality improvement,
- selective supplier evaluation,
- atau adjustment terhadap inventory strategy guna menekan potensi revenue leakage lebih lanjut.

#### **Expected Business Impact**

Implementasi return prevention strategy berpotensi:

- menurunkan revenue leakage,
- meningkatkan effective profitability,
- mengurangi operational burden akibat reverse logistics,
- serta meningkatkan customer satisfaction melalui product expectation alignment.

Pengurangan return rate bahkan dalam skala kecil dapat memberikan dampak signifikan terhadap **net revenue retention dan long-term business profitability.**

### 6.4 Operational Stability Strategy

#### **Key Finding**

Hasil analisis menunjukkan bahwa performa fulfillment TheLook relatif **stabil dan konsisten** dengan rata-rata fulfillment duration sekitar 3 hari sepanjang periode analisis.

Meskipun customer activity, transaction volume, dan revenue mengalami pertumbuhan yang sangat signifikan, tidak ditemukan indikasi operational bottleneck yang ekstrem pada proses fulfillment maupun delivery.

Selain itu, analisis hubungan antara fulfillment duration dan return behavior menunjukkan bahwa returned orders tidak memiliki fulfillment duration yang secara signifikan lebih lambat dibanding non-returned orders.

Hal ini mengindikasikan bahwa **fulfillment delay kemungkinan bukan faktor utama penyebab tingginya return behavior**.

#### **Business Implication**

Kondisi ini menunjukkan bahwa operasional fulfillment TheLook saat ini **berada dalam kondisi yang relatif healthy dan scalable**.

Perusahaan telah mampu mempertahankan consistency fulfillment performance meskipun bisnis mengalami pertumbuhan yang cukup agresif.

Temuan bahwa return behavior tidak berkorelasi kuat dengan fulfillment delay juga memberikan insight penting bahwa **prioritas improvement bisnis sebaiknya lebih difokuskan pada product experience dibanding percepatan operasional delivery semata**.

Namun demikian, seiring pertumbuhan bisnis yang terus meningkat, stabilitas operasional tetap perlu dipertahankan agar tidak berkembang menjadi bottleneck pada fase scaling berikutnya.

#### **Recommendation**

TheLook disarankan untuk: **mempertahankan operational consistency**.

Dengan tetap melakukan:

- monitoring fulfillment performance,
- capacity planning,
- dan operational forecasting

secara berkala seiring pertumbuhan business volume.

Karena fulfillment performance saat ini relatif stabil, perusahaan dapat:

- mengalihkan sebagian fokus improvement dari operational acceleration menuju customer experience optimization,
- terutama pada aspek product information, sizing accuracy, dan expectation management.

Selain itu, TheLook juga dapat mulai mengembangkan:

- operational early warning system,
- fulfillment performance dashboard,
- dan SLA monitoring

untuk menjaga scalability operasional dalam jangka panjang.

Untuk mendukung future business growth, perusahaan juga dapat mempertimbangkan:

- warehouse optimization,
- inventory distribution improvement,
- dan demand forecasting enhancement

agar operational stability tetap terjaga ketika transaction volume meningkat
lebih besar di masa mendatang.


#### **Expected Business Impact**

Implementasi strategi operational stability management berpotensi:

- menjaga fulfillment consistency,
- mempertahankan customer satisfaction,
- mengurangi risiko operational bottleneck,
- serta mendukung long-term business scalability.

Dengan mengalihkan fokus improvement ke customer experience dan return prevention, perusahaan juga dapat memperoleh **profitability improvement yang lebih optimal dibanding hanya berfokus pada percepatan delivery process**.